<a href="https://colab.research.google.com/github/satrishabh/1.1/blob/main/Agent_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
!pip install -q langchain langchain-google-genai requests

In [65]:
import os
from getpass import getpass

In [66]:
os.environ["GOOGLE_API_KEY"]=getpass("Enter your gemni key")

Enter your gemni key··········


In [67]:
import json
import requests
from typing  import List,Dict

In [68]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

In [69]:
leave_policy={
    "annual_leave_days":25,
    "sick_leave_days" : 15,
    "maternity_leave_days":10,
    "carry_forward_max":10
}

role_keyword={
    "data_engineer":["python","sql","airflow","spark","ETL","cloud"],
    "hr_generalist":["recruitment","onboarding","HRIS","compliance","employee_relations"]
}

In [70]:
# create a tool to calculate the resume score[tool- decorator,name, description, schema]
@tool
def hr_resume_scorer(role:str, resume_text:str)->str:
  """tool does a keyword match score for a given job role"""
  kws=role_keyword(role.lower())
  text=resume_text.lower()
  matched=[k for k in kws in text]
  score=int(round(len(matched)/max(len(kws),1))*100)
  missing=[k for k in kws if k not in text]

  return json.dumps({
      "role":role,
      "score":score,
      "missing_keywords":missing
  })

In [71]:
tools=[hr_resume_scorer]

In [90]:
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [82]:
system_prompt=(
    "you are an HR Assistant. \n"
    "rules:\n"
    "- USe a tool whever relevant.\n"
    "- keep answer short, structure and actionable.\n"
    "- If a question needs human intrvention, highlight that right away.\n"
)

In [85]:
from langgraph.prebuilt import create_react_agent

In [86]:
agent=create_react_agent(llm,tools)

/tmp/ipykernel_1556/2973680367.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent=create_react_agent(llm,tools)


In [87]:
def run_agent(user_prompt:str)->str:
  result=agent.invoke(
      {
          "messages":[
              SystemMessage(content=system_prompt),
              HumanMessage(content=user_prompt)
          ]
      }
  )
  return ["message"][-1].content

In [88]:
test_case_1=[{
    "prompt": """Evaluate this resume for a Data Engineer Role.
    Resume:
    Worked extensively with python, SQL and Spark,
    has build ETL pipelines and deployed them on AWS cloud.
    """
}]

In [91]:
for i in test_case_1:
  print(run_agent(i["prompt"]))

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}